# Diffusion Gauge on QWEN 3 Reasoning Traces

Demonstrates the manyLatents **diffusion gauge** on hidden-state activations
extracted from a QWEN 3 model during chain-of-thought math reasoning.

**Pipeline:** Load model → Generate CoT with hidden states → Segment into
reasoning steps → Pool hidden states per step → Apply diffusion gauge per
layer → Visualize cross-layer comparison.

**Requirements:** GPU node (A100 recommended). Run with `uv run jupyter lab`.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.manifold import MDS
from scipy import linalg

from manyagents.inference import (
    load_model,
    build_prompt,
    generate_with_hidden_states,
    segment,
    pool_hidden_states_per_step,
)
from manylatents.callbacks.diffusion_operator import DiffusionGauge

assert torch.cuda.is_available(), "This notebook requires a GPU"
device = "cuda"
MODEL_ID = "Qwen/Qwen3-4B"
print(f"Using {MODEL_ID} on {torch.cuda.get_device_name()}")

In [ ]:
model, tokenizer, _ = load_model(MODEL_ID, device_map="auto")
print(f"Loaded {MODEL_ID}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Hidden dim: {model.config.hidden_size}")

In [ ]:
MATH_PROBLEM = (
    "Find all integers n such that n^2 + 3n + 1 is a perfect square. "
    "Show your reasoning step by step."
)

# Backup problems if this one produces too few steps:
# "Prove that for any prime p > 3, p^2 - 1 is divisible by 24."
# "Find the last three digits of 7^2025."

prompt = build_prompt(tokenizer, MATH_PROBLEM)

print("Generating CoT (this may take a minute)...")
result = generate_with_hidden_states(
    model,
    tokenizer,
    prompt,
    max_new_tokens=2048,
    temperature=0.6,
    layers=None,  # capture ALL layers
)

print(f"Generated {result['n_new_tokens']} tokens in {result['generation_time_ms']}ms")
print(f"Hidden states shape: {result['token_hidden_states'].shape}")
print(f"  (n_tokens, n_layers, d_model)")
print()
print("--- Response ---")
print(result["text"][:2000])

In [ ]:
# Segment using <think> tag awareness (QWEN 3 emits <think>...</think>)
steps = segment(result["text"], tokenizer, segmentation="tags")

print(f"Found {len(steps)} reasoning steps:")
for i, step in enumerate(steps):
    kind = step["kind"]
    preview = step["text"][:80].replace("\n", " ")
    print(f"  [{i}] ({kind}) {preview}...")

# Pool token-level hidden states into per-step representations
pooled = pool_hidden_states_per_step(result["token_hidden_states"], steps)
n_steps, n_layers, d_model = pooled.shape
print(f"\nPooled shape: ({n_steps}, {n_layers}, {d_model})")
print(f"  = (n_steps, n_layers, d_model)")

# If too few steps (<3), suggest switching segmentation
if n_steps < 3:
    print("\nWarning: Very few steps detected. Try segmentation='delimiter' or a different problem.")